In [1]:
import pandas  as pd
import numpy as np
import biogeme.database as db
import biogeme.biogeme as bio
import biogeme.models as models
import biogeme.expressions as exp
import biogeme.results as res

import seaborn as sns
import matplotlib.pyplot as plt

import re
import os
import copy
import warnings
import functools
import contextlib
import time

from decimal import Decimal
from datetime import datetime
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks

from sklearn.metrics import confusion_matrix, classification_report, fbeta_score


In [2]:
plt.style.use('dark_background')
pd.set_option("display.precision", 2)
seed = 1
np.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

In [3]:
# Data Loading
df = pd.read_csv('data_incentive_attitudes_residencedata.csv')

# Some basic data preprocessing steps
df = df.drop(['X', 'trip','value','long_o', 'lat_o', 'long_d.x', 
              'lat_d.x', 'pincode', 'address','Prefecture', 'City', 
              'Town','lat_d.return','long_d.return', 'dest_inside'], axis='columns')
df = df.rename(columns={'pubcost':'traincost', 
                   'pubtime':'traintime', 
                   'pubavail':'trainavail',
                    'bicycleincentive':'bikeincentive'})

# Replacing 'pub' with 'train' in `mode` column.
selected_mode = df['mode'].to_numpy()
df['mode'] = np.where(selected_mode=='pub', 'train', selected_mode)

purpose_code_dict = {100: 'Commuting to work / school',
                    101: 'Go Home',
                    200: 'Shopping for daily necessities',
                    201: 'Shopping other than daily necessities',
                    202: 'Meals and entertainment',
                    300: 'business',
                    400: 'Outpatient',
                    500: 'Pick-up and drop-off',
                    600: 'Sightseeing / Leisure',
                    998: 'others'}

attitudinal_variables = [x for x in df.columns if bool(re.search(r'\d', x))]
ohe_generic_variables = dict()

df['walkavail'] = np.where(df['mode']=='walk', 1, np.where(df['cardistance']<=7, 1, 0))
df['busincentive'] = np.where(df['incentivezone']==1, df['busincentive'], 0)
df['carincentive'] = np.where(df['incentivezone']==1, df['carincentive'], 0)
df['trainincentive'] = np.where(df['incentivezone']==1, df['trainincentive'], 0)
df['bikeincentive'] = np.where(df['incentivezone']==1, df['bikeincentive'], 0)
df['walkincentive'] = np.where(df['incentivezone']==1, df['walkincentive'], 0)
df['motorincentive'] = np.where(df['incentivezone']==1, df['motorincentive'], 0)


In [4]:
# utility functions for extracting time-zones and trip duration information from arrival and departure times of a trip.

def get_trip_duration(df):
    """
    Returns the total minutes a trip lasted given its departure and arrival datetimes.
    """
    dep_datetime = datetime.strptime(df['departure_time'], '%m/%d/%Y %H:%M')
    arr_datetime = datetime.strptime(df['arrival_time'], '%m/%d/%Y %H:%M')
    
    return int((arr_datetime - dep_datetime).total_seconds()/60)

def get_day_zones(trip_datetime):
    """
    Divides the time into 4 zones and ordinally encoding them.
    Early morning (00:00 to 6:00) - 1
    AM peak (6:00 to 10:00) - 2
    Off peak (10:00 to 16:00) - 3
    PM peak (16:00 to 20:00) - 4
    Evening (20:00 to 00:00) - 5
    """
    dep_hrs, dep_mins = [int(val) for val in trip_datetime.split(' ')[1].split(':')]
    if dep_hrs >= 0 and dep_hrs < 6:
        return 1
    elif dep_hrs >= 6 and dep_hrs < 10:
        return 2
    elif dep_hrs >= 10 and dep_hrs < 16:
        return 3
    elif dep_hrs >= 16 and dep_hrs < 20:
        return 4
    else:
        return 5

ohe = OneHotEncoder()
df['high_income'] = np.where(df['INCOME'] >= 5, 1, 0)

# `job_type` column
job_type_sparse_matrix = ohe.fit_transform(df['job_type'].to_numpy().reshape(-1, 1)).toarray()
job_type_column_names = ['job_type_'+job for job in df['job_type'].unique()]
job_type_df = pd.DataFrame(data=job_type_sparse_matrix, columns=job_type_column_names)
job_type_df = job_type_df.drop(job_type_df.columns[0], axis=1)
ohe_generic_variables['jobs'] = list(job_type_df.columns)

# `information` column
info_sparse_matrix = ohe.fit_transform(df['information'].to_numpy().reshape(-1, 1)).toarray()
info_df = pd.DataFrame(data=info_sparse_matrix, columns=ohe.categories_[0])
info_df = info_df.drop(info_df.columns[0], axis=1)
ohe_generic_variables['info'] = list(info_df.columns)

# `SEX` column
SEX_sparse_matrix = ohe.fit_transform(df['SEX'].to_numpy().reshape(-1, 1)).toarray()
SEX_df = pd.DataFrame(data=SEX_sparse_matrix, columns=['Male', 'Female'])
SEX_df = SEX_df.drop(SEX_df.columns[0], axis=1)

# `Purpose` column
Purpose_sparse_matrix = ohe.fit_transform(df['Purpose'].to_numpy().reshape(-1, 1)).toarray()
Purpose_column_names = ['Purpose_'+str(purpose_code) for purpose_code in df['Purpose'].unique()]
Purpose_df = pd.DataFrame(data=Purpose_sparse_matrix, columns=Purpose_column_names)
Purpose_df = Purpose_df.drop('Purpose_999', axis=1)
ohe_generic_variables['purposes'] = list(Purpose_df.columns)

# Getting the trip duration in minutes
df['trip_duration'] = df[['departure_time', 'arrival_time']].T.apply(get_trip_duration)

# Converting departure time in time zones and one-hot encoding it
dep_time_zones_sparse_matrix = ohe.fit_transform(df['departure_time'].apply(get_day_zones).to_numpy().reshape(-1, 1)).toarray()
dep_time_zones_names = ['Early_morning_departure', 'AM_peak_departure', 'Off_peak_departure', 'PM_peak_departure', 'Night_departure']
dep_time_zones_df = pd.DataFrame(data=dep_time_zones_sparse_matrix, columns=dep_time_zones_names)
dep_time_zones_df = dep_time_zones_df.drop('Night_departure', axis=1)
ohe_generic_variables['departures'] = list(dep_time_zones_df.columns)

# combining all the one-hot encoded dataframes with the main dataframe
df2 = pd.concat([df, dep_time_zones_df, Purpose_df, SEX_df, job_type_df, info_df], axis=1).drop(
    ['departure_time', 'arrival_time', 'Purpose', 'SEX', 'job_type'], axis='columns')

# Removing some unnecessary columns
df2 = df2.drop(['user_id', 'trip_id', 'recco', 'information', 'incentivezone', 'task', 'income_con'], axis='columns')

df2.head()

,buscost,bustime,carcost,cartime,cardistance,traincost,traintime,walktime,walkcost,biketime,...,job_type_Part time job,job_type_Management executive,job_type_civil servant,job_type_Housewife,job_type_Self employed/ Freelance,job_type_Others,job_type_Unemployed,no info,only enviro,only health
0,770,69.4,151.7,43.72,15.17,240,60.9,182.04,0,60.68,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,690,66.2,128.6,38.96,12.86,240,44.9,154.32,0,51.44,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,770,69.4,151.7,43.72,15.17,240,60.9,182.04,0,60.68,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3,500,56.2,118.5,29.53,11.85,400,59.0,142.20,0,47.40,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
4,770,69.4,151.7,43.72,15.17,240,60.9,182.04,0,60.68,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0


In [5]:
def get_CBD_data_split(df, drop_att_vars=True, apply_smote=False, apply_tomek_links=False,
                      apply_standardization=False, cols_to_standardize=[]):
    if drop_att_vars:
        df = df.drop(attitudinal_variables, axis='columns')
        
        oCBD_df = df.loc[df['address_inside'] == False]
        iCBD_df = df.loc[(df['address_inside'] == True)]
        
        label_mapping_dict = {'bus': 1, 'car': 2, 'train': 3, 'walk': 4, 'bike': 5, 'motor': 6}
                
        y_oCBD = np.array([label_mapping_dict[label] for label in oCBD_df['mode']])
        X_oCBD = oCBD_df.drop('mode', axis='columns')
        y_iCBD = np.array([label_mapping_dict[label] for label in iCBD_df['mode']])
        X_iCBD = iCBD_df.drop('mode', axis='columns')
        
        if apply_smote:
            smote = SMOTE(random_state=500)
            X_oCBD, y_oCBD = smote.fit_resample(X_oCBD, y_oCBD)
            X_iCBD, y_iCBD = smote.fit_resample(X_iCBD, y_iCBD)
        
        if apply_tomek_links:
            tomek = TomekLinks()
            X_oCBD, y_oCBD = tomek.fit_resample(X_oCBD, y_oCBD)
            X_iCBD, y_iCBD = tomek.fit_resample(X_iCBD, y_iCBD)
        
        if apply_standardization:
            ss =  StandardScaler()
            X_oCBD_scaled_arr = ss.fit_transform(X_oCBD[cols_to_standardize])
            X_iCBD_scaled_arr = ss.transform(X_iCBD[cols_to_standardize])
            X_oCBD_scaled = pd.DataFrame(X_oCBD_scaled_arr, columns=cols_to_standardize)
            X_iCBD_scaled = pd.DataFrame(X_iCBD_scaled_arr, columns=cols_to_standardize)
            X_oCBD_scaled.set_index(X_oCBD.index, inplace=True)
            X_iCBD_scaled.set_index(X_iCBD.index, inplace=True)
            X_oCBD.iloc[0:X_oCBD.shape[0], X_oCBD.columns.get_indexer(cols_to_standardize)] = X_oCBD_scaled
            X_iCBD.iloc[0:X_iCBD.shape[0], X_iCBD.columns.get_indexer(cols_to_standardize)] = X_iCBD_scaled
        
        return ((X_oCBD, y_oCBD), (X_iCBD, y_iCBD)), label_mapping_dict
    

In [7]:
output_categories = ['bus', 'car', 'train', 'walk', 'bike', 'motor']

asc_variable_names = {
    'bus': ['buscost', 'bustime', 'busincentive'],
    'car': ['carcost', 'cartime', 'carincentive'],
    'train': ['traincost', 'traintime', 'trainincentive'], 
    'walk': ['walkcost', 'walktime', 'walkincentive'],
    'bike': [ 'bikecost', 'biketime', 'bikeincentive'],
    'motor': ['motorcost', 'motortime', 'motorincentive']
}

avail_variable_names = {
    'bus': 'busavail',
    'train': 'trainavail',
    'walk': 'walkavail',
    'bike': 'bikeavail',
    'motor': 'motoravail'
}

In [8]:
((X_oCBD, y_oCBD), (X_iCBD, y_iCBD)), label_mapping_dict = get_CBD_data_split(df2, drop_att_vars=True)

In [9]:
X_oCBD['mode'] = y_oCBD
X_oCBD['caravail'] = np.ones(X_oCBD.shape[0])
X_iCBD['mode'] = y_iCBD
X_iCBD['caravail'] = np.ones(X_iCBD.shape[0])

In [10]:
bio_db = db.Database("Outside CBD", X_oCBD.drop('address_inside', axis='columns'))
bio_db_test = db.Database("Inside CBD", X_iCBD.drop('address_inside', axis='columns'))

In [11]:
globals().update(bio_db.variables)

In [12]:
ind_spec_vars = ['AGE', 'Female', 'high_income', 'nearest_bus_dist', 'nearest_train_dist',]
ind_spec_vars = []

In [14]:
ASC_BUS   = exp.Beta('ASC_BUS',0,None ,None ,0)
ASC_TRAIN = exp.Beta('ASC_TRAIN',0,None ,None ,0)
ASC_WALK  = exp.Beta('ASC_WALK',0,None ,None ,0)
ASC_BIKE = exp.Beta('ASC_BIKE', 0, None, None, 0)
ASC_MOTOR = exp.Beta('ASC_MOTOR', 0, None, None, 0)

B_TIME    = exp.Beta('B_TIME',0,None ,None ,0)
B_COST    = exp.Beta('B_COST',0,None ,None ,0)
B_INC    = exp.Beta('B_INC',0,None ,None ,0)

B_IS = {}
for ind_spec_var in ind_spec_vars:
    if ind_spec_var == 'nearest_bus_dist':
        B_IS[f'B_{ind_spec_var}_bus'] = exp.Beta(f'B_{ind_spec_var}_bus', 0, None, None, 0)
    elif ind_spec_var == 'nearest_train_dist':
        B_IS[f'B_{ind_spec_var}_train'] = exp.Beta(f'B_{ind_spec_var}_train', 0, None, None, 0)
    else:
        for alternative in output_categories:
            if alternative == 'car':
                continue
            B_IS[f'B_{ind_spec_var}_{alternative}'] = exp.Beta(f'B_{ind_spec_var}_{alternative}', 0, None, None, 0)

In [15]:
U_CAR = B_TIME*cartime + B_COST*carcost + B_INC*carincentive
U_BUS = B_TIME*bustime + B_COST*buscost + B_INC*busincentive + ASC_BUS
U_TRAIN = B_TIME*traintime + B_COST*traincost + B_INC*trainincentive + ASC_TRAIN
U_WALK = B_TIME*walktime + B_COST*walkcost + B_INC*walkincentive + ASC_WALK
U_BIKE = B_TIME*biketime + B_COST*bikecost + B_INC*bikeincentive + ASC_BIKE
U_MOTOR = B_TIME*motortime + B_COST*motorcost + B_INC*motorincentive + ASC_MOTOR

for is_var in ind_spec_vars:
    if is_var == 'nearest_bus_dist':
        U_BUS += nearest_bus_dist*B_IS[f'B_{is_var}_bus']
    elif is_var == 'nearest_train_dist':
        U_TRAIN += nearest_train_dist*B_IS[f'B_{is_var}_train']
    else:
        U_BUS += bio_db.variables[is_var] * B_IS[f'B_{is_var}_bus']
        U_TRAIN += bio_db.variables[is_var] * B_IS[f'B_{is_var}_train']
        U_WALK += bio_db.variables[is_var] * B_IS[f'B_{is_var}_walk']
        U_BIKE += bio_db.variables[is_var] * B_IS[f'B_{is_var}_bike']
        U_MOTOR += bio_db.variables[is_var] * B_IS[f'B_{is_var}_motor']

In [17]:
utilities = {
    1 : U_BUS,
    2 : U_CAR,
    3 : U_TRAIN,
    4 : U_WALK,
    5 : U_BIKE,
    6 : U_MOTOR
}

avails = {
    1 : busavail,
    2 : caravail,
    3 : trainavail,
    4 : walkavail,
    5 : bikeavail,
    6 : motoravail
}

In [19]:
avail_choice_ratio_dict_train = {output_categories[int(key)-1]: np.round(value[1]/value[0], 2) for key, value in bio_db.choiceAvailabilityStatistics(avails, mode).items()}
print(f"Training data's mode availability vs choice ratio: \n{avail_choice_ratio_dict_train}\n")

avail_choice_ratio_dict_test = {output_categories[int(key)-1]: np.round(value[1]/value[0], 2) for key, value in bio_db_test.choiceAvailabilityStatistics(avails, mode).items()}
print(f"Testing data's mode availability vs choice ratio: \n{avail_choice_ratio_dict_test}")

Training data's mode availability vs choice ratio: 
{'bus': 12.48, 'car': 1.43, 'train': 7.69, 'walk': 5.05, 'bike': 3.99, 'motor': 4.75}

Testing data's mode availability vs choice ratio: 
{'bus': 14.57, 'car': 1.79, 'train': 11.58, 'walk': 7.17, 'bike': 3.07, 'motor': 1.99}


In [20]:
logprob = models.loglogit(utilities, avails, mode)
model_note = "Using only travel time and cost variables"
biogeme  = bio.BIOGEME(bio_db, logprob)
biogeme.algorithm_name = 'LS-BFGS'
biogeme.tolerance = 1e-8
biogeme.maxiter = 1000
biogeme.modelName = 'MNL_model'
#biogeme.generate_html = True
biogeme.only_robust_stats = False

results = biogeme.estimate()

print(f"HTML file:    {results.data.htmlFileName}")

HTML file:    MNL_model~01.html


In [71]:
f,g,h,gdiff, hdiff = biogeme.checkDerivatives(list(results.getBetaValues().values()))
g

array([ 1.59535611e-08, -2.20889209e-08,  1.47584607e-08,  3.77495484e-08,
       -7.60527401e-08, -1.22251640e-06, -2.08074698e-06, -9.21853598e-08])

In [68]:
dict(zip(list(results.getBetaValues().keys()), g)) 

{'ASC_BIKE': 1.5953561138815076e-08,
 'ASC_BUS': -2.208892091459802e-08,
 'ASC_MOTOR': 1.4758460675778906e-08,
 'ASC_TRAIN': 3.774954837609812e-08,
 'ASC_WALK': -7.605274010558105e-08,
 'B_COST': -1.2225164027768187e-06,
 'B_INC': -2.080746980936965e-06,
 'B_TIME': -9.218535979016451e-08}

In [80]:
betas = results.getBetaValues()
betas['B_COST'] = -0.00243883
betas['B_TIME'] = -0.05643418
betas['B_INC'] = 0.00363528
betas['ASC_BUS'] = -0.5311894
betas['ASC_TRAIN'] = -0.86371666
betas['ASC_WALK'] = 0.12339422
betas['ASC_BIKE'] = -0.6833264
betas['ASC_MOTOR']  = -1.1309592 
for k,v in betas.items():
    print(f"{k:10}=\t{v:.10g}")

betas_vector = [v for k, v in betas.items()]
betas_vector

ASC_BIKE  =	-0.6833264
ASC_BUS   =	-0.5311894
ASC_MOTOR =	-1.1309592
ASC_TRAIN =	-0.86371666
ASC_WALK  =	0.12339422
B_COST    =	-0.00243883
B_INC     =	0.00363528
B_TIME    =	-0.05643418


[-0.6833264,
 -0.5311894,
 -1.1309592,
 -0.86371666,
 0.12339422,
 -0.00243883,
 0.00363528,
 -0.05643418]

In [81]:
f, g, h, bh = biogeme.calculateLikelihoodAndDerivatives(betas_vector, scaled=False)
g


array([  3.6584434 ,   0.3230647 ,  -0.15499331,  -6.4979469 ,
       -25.06472908,  49.25530952,  50.68940701,   3.02145999])

In [76]:
f

-1306.1485612668894

In [20]:
print(results.shortSummary())

Results for model MNL_model
Nbr of parameters:		25
Sample size:			1851
Excluded data:			0
Final log likelihood:		-1204.622
Akaike Information Criterion:	2459.243
Bayesian Information Criterion:	2597.331



In [26]:
new_indices = ['B_COST',
 'B_TIME',
 'B_INC',
 'ASC_BUS',
 'ASC_TRAIN',
 'ASC_WALK',
 'ASC_BIKE',
 'ASC_MOTOR',
 'B_AGE_bus',
 'B_AGE_train',
 'B_AGE_walk',
 'B_AGE_bike',
 'B_AGE_motor',
 'B_Female_bus',
 'B_Female_train',
 'B_Female_walk',
 'B_Female_bike',
 'B_Female_motor',
 'B_high_income_bus',
 'B_high_income_train',
 'B_high_income_walk',
 'B_high_income_bike',
 'B_high_income_motor',
 'B_nearest_bus_dist_bus',
 'B_nearest_train_dist_train']

In [29]:
results.getEstimatedParameters(onlyRobust=False)

,Value,Std err,t-test,p-value,Rob. Std err,Rob. t-test,Rob. p-value
ASC_BIKE,-2.07e+00,8.60e-01,-2.40e+00,1.62e-02,7.06e-01,-2.93e+00,3.43e-03
ASC_BUS,-2.28e+00,6.66e-01,-3.43e+00,6.09e-04,6.67e-01,-3.42e+00,6.19e-04
ASC_MOTOR,4.16e+00,1.35e+00,3.08e+00,2.05e-03,1.37e+00,3.04e+00,2.39e-03
ASC_TRAIN,-5.36e-01,4.42e-01,-1.21e+00,2.25e-01,5.09e-01,-1.05e+00,2.93e-01
ASC_WALK,2.92e-01,4.91e-01,5.95e-01,5.52e-01,4.99e-01,5.86e-01,5.58e-01
B_AGE_bike,2.73e-02,1.98e-02,1.38e+00,1.67e-01,1.65e-02,1.66e+00,9.76e-02
B_AGE_bus,2.27e-02,1.30e-02,1.74e+00,8.14e-02,1.32e-02,1.72e+00,8.55e-02
B_AGE_motor,-9.36e-02,2.59e-02,-3.62e+00,2.99e-04,2.68e-02,-3.50e+00,4.74e-04
B_AGE_train,-7.69e-03,8.61e-03,-8.94e-01,3.72e-01,9.98e-03,-7.71e-01,4.41e-01
B_AGE_walk,-6.27e-03,1.00e-02,-6.26e-01,5.31e-01,1.01e-02,-6.21e-01,5.34e-01


In [30]:
results.getVarCovar()

,ASC_BIKE,ASC_BUS,ASC_MOTOR,ASC_TRAIN,ASC_WALK,B_AGE_bike,B_AGE_bus,B_AGE_motor,B_AGE_train,B_AGE_walk,...,B_Female_walk,B_INC,B_TIME,B_high_income_bike,B_high_income_bus,B_high_income_motor,B_high_income_train,B_high_income_walk,B_nearest_bus_dist_bus,B_nearest_train_dist_train
ASC_BIKE,0.74,0.03,-0.0,0.04,0.05,-0.02,-0.0,-0.0,-0.0,-0.0,...,-0.01,-0.0,-0.0,0.14,0.0,-0.0,0.0,0.0,0.0,-0.0
ASC_BUS,0.03,0.44,0.02,0.06,0.05,-0.0,-0.01,-0.0,-0.0,-0.0,...,-0.01,-0.0,-0.0,-0.0,0.03,-0.0,0.0,0.0,-0.0,0.0
ASC_MOTOR,-0.0,0.02,1.82,0.03,0.02,0.0,-0.0,-0.03,-0.0,-0.0,...,-0.01,0.0,0.0,-0.02,-0.0,0.0,0.0,0.0,-0.0,-0.0
ASC_TRAIN,0.04,0.06,0.03,0.2,0.03,-0.0,-0.0,-0.0,-0.0,-0.0,...,-0.01,-0.0,-0.0,0.0,0.01,0.0,0.01,0.0,0.0,-0.0
ASC_WALK,0.05,0.05,0.02,0.03,0.24,-0.0,-0.0,-0.0,-0.0,-0.0,...,-0.05,-0.0,-0.0,0.01,0.01,0.0,0.0,0.02,0.0,0.0
B_AGE_bike,-0.02,-0.0,0.0,-0.0,-0.0,0.0,0.0,-0.0,0.0,0.0,...,0.0,0.0,0.0,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,0.0
B_AGE_bus,-0.0,-0.01,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0
B_AGE_motor,-0.0,-0.0,-0.03,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,...,0.0,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0
B_AGE_train,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,0.0
B_AGE_walk,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0


In [31]:
results.getBetaValues

<bound method bioResults.getBetaValues of <biogeme.results.bioResults object at 0x000002095C329D90>>

In [32]:
gs = results.getGeneralStatistics()

for k,v in gs.items():
    print("{}= {}".format(k.ljust(45),v[0]))

Number of estimated parameters               = 25
Sample size                                  = 1851
Excluded observations                        = 0
Init log likelihood                          = -1204.6217428438094
Final log likelihood                         = -1204.6217428438094
Likelihood ratio test for the init. model    = -0.0
Rho-square for the init. model               = 0.0
Rho-square-bar for the init. model           = -0.02075340259173908
Akaike Information Criterion                 = 2459.243485687619
Bayesian Information Criterion               = 2597.3305185019562
Final gradient norm                          = 7.268739056046535e-06
Nbr of threads                               = 8


In [33]:
prob_bus = models.logit(utilities, avails, 1)
prob_car = models.logit(utilities, avails, 2)
prob_train = models.logit(utilities, avails, 3)
prob_walk = models.logit(utilities, avails, 4)
prob_bike = models.logit(utilities, avails, 5)
prob_motor = models.logit(utilities, avails, 6)

In [34]:
simulate ={
        1:  prob_bus,
        2: prob_car,
        3:  prob_train,
        4: prob_walk,
        5: prob_bike,
        6: prob_motor
          }

In [35]:
biogeme = bio.BIOGEME(bio_db_test, simulate)
biogeme.modelName = "MNL_model_test"
betas = biogeme.freeBetaNames()

print('Extracting the following variables:')
for k in betas:
    print('\t',k)

results = res.bioResults(pickleFile='MNL_model.pickle')
betaValues = results.getBetaValues ()

The chosen alternative [`1.0`] is not available for the following observations (rownumber[choice]): 0[1.0]-1[1.0]-2[1.0]-3[1.0]-4[1.0]-5[1.0]-6[1.0]-7[1.0]-8[1.0]-9[1.0]-10[1.0]-11[1.0]-13[1.0]-14[1.0...
The chosen alternative [`3.0`] is not available for the following observations (rownumber[choice]): 0[3.0]-1[3.0]-2[3.0]-3[3.0]-4[3.0]-5[3.0]-6[3.0]-7[3.0]-8[3.0]-9[3.0]-10[3.0]-11[3.0]-13[3.0]-14[3.0...
The chosen alternative [`4.0`] is not available for the following observations (rownumber[choice]): 4[4.0]-11[4.0]-15[4.0]-26[4.0]-27[4.0]-30[4.0]-31[4.0]-32[4.0]-33[4.0]-34[4.0]-35[4.0]-36[4.0]-37[4....
The chosen alternative [`5.0`] is not available for the following observations (rownumber[choice]): 0[5.0]-1[5.0]-2[5.0]-3[5.0]-4[5.0]-5[5.0]-6[5.0]-7[5.0]-8[5.0]-9[5.0]-10[5.0]-11[5.0]-12[5.0]-13[5.0...
The chosen alternative [`6.0`] is not available for the following observations (rownumber[choice]): 0[6.0]-1[6.0]-2[6.0]-3[6.0]-4[6.0]-5[6.0]-6[6.0]-7[6.0]-8[6.0]-9[6.0]-10[6.0]-11[6.0

Extracting the following variables:
	 ASC_BIKE
	 ASC_BUS
	 ASC_MOTOR
	 ASC_TRAIN
	 ASC_WALK
	 B_AGE_bike
	 B_AGE_bus
	 B_AGE_motor
	 B_AGE_train
	 B_AGE_walk
	 B_COST
	 B_Female_bike
	 B_Female_bus
	 B_Female_motor
	 B_Female_train
	 B_Female_walk
	 B_INC
	 B_TIME
	 B_high_income_bike
	 B_high_income_bus
	 B_high_income_motor
	 B_high_income_train
	 B_high_income_walk
	 B_nearest_bus_dist_bus
	 B_nearest_train_dist_train


In [36]:
simulatedValues = biogeme.simulate(betaValues)
print(simulatedValues.head())

The chosen alternative [`1.0`] is not available for the following observations (rownumber[choice]): 0[1.0]-1[1.0]-2[1.0]-3[1.0]-4[1.0]-5[1.0]-6[1.0]-7[1.0]-8[1.0]-9[1.0]-10[1.0]-11[1.0]-13[1.0]-14[1.0...
The chosen alternative [`3.0`] is not available for the following observations (rownumber[choice]): 0[3.0]-1[3.0]-2[3.0]-3[3.0]-4[3.0]-5[3.0]-6[3.0]-7[3.0]-8[3.0]-9[3.0]-10[3.0]-11[3.0]-13[3.0]-14[3.0...
The chosen alternative [`4.0`] is not available for the following observations (rownumber[choice]): 4[4.0]-11[4.0]-15[4.0]-26[4.0]-27[4.0]-30[4.0]-31[4.0]-32[4.0]-33[4.0]-34[4.0]-35[4.0]-36[4.0]-37[4....
The chosen alternative [`5.0`] is not available for the following observations (rownumber[choice]): 0[5.0]-1[5.0]-2[5.0]-3[5.0]-4[5.0]-5[5.0]-6[5.0]-7[5.0]-8[5.0]-9[5.0]-10[5.0]-11[5.0]-12[5.0]-13[5.0...
The chosen alternative [`6.0`] is not available for the following observations (rownumber[choice]): 0[6.0]-1[6.0]-2[6.0]-3[6.0]-4[6.0]-5[6.0]-6[6.0]-7[6.0]-8[6.0]-9[6.0]-10[6.0]-11[6.0

      1     2    3     4    5    6
16  0.0  0.85  0.0  0.15  0.0  0.0
17  0.0  0.85  0.0  0.15  0.0  0.0
18  0.0  0.85  0.0  0.15  0.0  0.0
19  0.0  0.85  0.0  0.15  0.0  0.0
20  0.0  1.00  0.0  0.00  0.0  0.0


In [37]:
prob_max_test = simulatedValues.idxmax(axis=1)

In [38]:
biogeme = bio.BIOGEME(bio_db, simulate)
biogeme.modelName = "MNL_model_test"
betas = biogeme.freeBetaNames()

print('Extracting the following variables:')
for k in betas:
    print('\t',k)

results = res.bioResults(pickleFile='MNL_model.pickle')
betaValues = results.getBetaValues ()

The chosen alternative [`1.0`] is not available for the following observations (rownumber[choice]): 3[1.0]-16[1.0]-18[1.0]-19[1.0]-20[1.0]-25[1.0]-26[1.0]-27[1.0]-29[1.0]-31[1.0]-32[1.0]-34[1.0]-35[1....
The chosen alternative [`3.0`] is not available for the following observations (rownumber[choice]): 3[3.0]-16[3.0]-18[3.0]-19[3.0]-20[3.0]-25[3.0]-26[3.0]-27[3.0]-29[3.0]-31[3.0]-32[3.0]-34[3.0]-35[3....
The chosen alternative [`4.0`] is not available for the following observations (rownumber[choice]): 0[4.0]-1[4.0]-2[4.0]-3[4.0]-4[4.0]-5[4.0]-6[4.0]-7[4.0]-8[4.0]-9[4.0]-10[4.0]-11[4.0]-12[4.0]-13[4.0...
The chosen alternative [`5.0`] is not available for the following observations (rownumber[choice]): 0[5.0]-1[5.0]-2[5.0]-3[5.0]-4[5.0]-5[5.0]-6[5.0]-7[5.0]-8[5.0]-9[5.0]-10[5.0]-11[5.0]-12[5.0]-13[5.0...
The chosen alternative [`6.0`] is not available for the following observations (rownumber[choice]): 0[6.0]-1[6.0]-2[6.0]-3[6.0]-4[6.0]-5[6.0]-6[6.0]-7[6.0]-8[6.0]-9[6.0]-10[6.0]-11[6.0

Extracting the following variables:
	 ASC_BIKE
	 ASC_BUS
	 ASC_MOTOR
	 ASC_TRAIN
	 ASC_WALK
	 B_AGE_bike
	 B_AGE_bus
	 B_AGE_motor
	 B_AGE_train
	 B_AGE_walk
	 B_COST
	 B_Female_bike
	 B_Female_bus
	 B_Female_motor
	 B_Female_train
	 B_Female_walk
	 B_INC
	 B_TIME
	 B_high_income_bike
	 B_high_income_bus
	 B_high_income_motor
	 B_high_income_train
	 B_high_income_walk
	 B_nearest_bus_dist_bus
	 B_nearest_train_dist_train


In [39]:
simulatedValues = biogeme.simulate(betaValues)
print(simulatedValues.head())
prob_max_train = simulatedValues.idxmax(axis=1)

The chosen alternative [`1.0`] is not available for the following observations (rownumber[choice]): 3[1.0]-16[1.0]-18[1.0]-19[1.0]-20[1.0]-25[1.0]-26[1.0]-27[1.0]-29[1.0]-31[1.0]-32[1.0]-34[1.0]-35[1....
The chosen alternative [`3.0`] is not available for the following observations (rownumber[choice]): 3[3.0]-16[3.0]-18[3.0]-19[3.0]-20[3.0]-25[3.0]-26[3.0]-27[3.0]-29[3.0]-31[3.0]-32[3.0]-34[3.0]-35[3....
The chosen alternative [`4.0`] is not available for the following observations (rownumber[choice]): 0[4.0]-1[4.0]-2[4.0]-3[4.0]-4[4.0]-5[4.0]-6[4.0]-7[4.0]-8[4.0]-9[4.0]-10[4.0]-11[4.0]-12[4.0]-13[4.0...
The chosen alternative [`5.0`] is not available for the following observations (rownumber[choice]): 0[5.0]-1[5.0]-2[5.0]-3[5.0]-4[5.0]-5[5.0]-6[5.0]-7[5.0]-8[5.0]-9[5.0]-10[5.0]-11[5.0]-12[5.0]-13[5.0...
The chosen alternative [`6.0`] is not available for the following observations (rownumber[choice]): 0[6.0]-1[6.0]-2[6.0]-3[6.0]-4[6.0]-5[6.0]-6[6.0]-7[6.0]-8[6.0]-9[6.0]-10[6.0]-11[6.0

      1     2     3    4    5    6
0  0.02  0.85  0.14  0.0  0.0  0.0
1  0.02  0.79  0.20  0.0  0.0  0.0
2  0.02  0.85  0.14  0.0  0.0  0.0
3  0.00  1.00  0.00  0.0  0.0  0.0
4  0.02  0.85  0.14  0.0  0.0  0.0


In [41]:
print(f"Training score: {fbeta_score(y_oCBD, prob_max_train, beta=2, average='weighted'):.3f}")

Training score: 0.708


In [42]:
print(f"Testing score: {fbeta_score(y_iCBD, prob_max_test, beta=2, average='weighted'):.3f}")

Testing score: 0.518
